In [31]:
import pandas as pd
import re
from pathlib import Path
from functools import reduce
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed


### Contents
1. Import list of SNPs
2. Select tissues of interest and apply process_snps_gtex to obtain the tissue-specific eQTL expression profile
3. Format data for UP SET plot


- After importing the list of 21 variants obtained from the LD clumping analysis (clump_results_by_tissue/clump_index_snps_MERGED.tsv), the tissues of interest were selected. For each tissue, the GTEx Portal was queried again to retrieve the corresponding tissue-specific eQTL expression profile for the selected variants.
- The gtex_lookup_snp() function was applied using p-value cutoffs of 0.05 and 1 to retrieve variant–tissue associations. The cutoff     of 1 was used to capture all available variants independently of association significance. For each tissue, the effective allele      alignment was verified to ensure correct interpretation of the direction of effect.


In [34]:


def gtex_lookup_snp(
    gencodeId,
    snp_id,
    tissues,
    dataset="gtex_v8",
    p_cutoff=1,#0.05,
    max_workers=20
):
    """
    Query GTEx for a single SNP across all tissues.
    Returns:
        dict:
           {
             "snp_id": snp_id,
             "gencodeId": gencodeId,
             "significant_tissues": {
                 tissue_name: {maf, nes, pValue, ...}
             }
           }
    """

    base_url = "https://gtexportal.org/api/v2/association/dyneqtl"
    session = requests.Session()

    result = {
        "snp_id": snp_id,
        "gencodeId": gencodeId,
        "significant_tissues": {}
    }

    def fetch_one(tissue):
        params = {
            "gencodeId": gencodeId,
            "variantId": snp_id,
            "datasetId": dataset,
            "tissueSiteDetailId": tissue
        }
        try:
            r = session.get(base_url, params=params, timeout=5)
            if r.status_code != 200:
                return None

            data = r.json()
            pval = data.get("pValue")
            if pval is None:
                return None

            if pval <= p_cutoff:
                return tissue, {
                    "geneSymbol": data.get("geneSymbol"),
                    "maf": data.get("maf"),
                    "nes": data.get("nes"),
                    "pValue": pval,
                    "tStatistic": data.get("tStatistic"),
                    "variantId": data.get("variantId")
                }

        except Exception:
            return None

        return None

    # Parallel execution
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(fetch_one, t) for t in tissues]

        for fut in as_completed(futures):
            r = fut.result()
            if r is not None:
                tissue, values = r
                result["significant_tissues"][tissue] = values

    return result
    
def prepare_plink_input(df):
    """
    Preprocess SNPs for PLINK clumping:
    1. Parse variantId to extract CHR, BP, ref, alt
    2. Remove multi-allelic and ambiguous SNPs
    3. Orient alleles so ALT = expression-increasing
    4. Prepare final PLINK input
    
    Parameters
    ----------
    df : it must contain columns: variantId, nes, pValue, snp_id, tissue

    Returns
    -------
    plink_df : PLINK-ready DataFrame with columns: ID, CHR, BP, P, REF, ALT, snp_id, tissue
    """
    
    # -----------------------------
    # 1. Parse variantId
    # -----------------------------
    def parse_variant(v):
        m = re.match(r"chr(\d+|X|Y|MT)_(\d+)_([ACGT]+)_([ACGT]+)_b38", v)
        if m:
            return pd.Series({
                "CHR": m.group(1),
                "BP": int(m.group(2)),
                "ref": m.group(3),
                "alt": m.group(4)
            })
        else:
            return pd.Series({"CHR": None, "BP": None, "ref": None, "alt": None})
    
    df[["CHR", "BP", "ref", "alt"]] = df["variantId"].apply(parse_variant)
    
    # -----------------------------
    # 2. Filter valid biallelic SNPs
    # -----------------------------
    valid_alleles = {"A", "C", "G", "T"}
    ambiguous_pairs = {("A","T"), ("T","A"), ("C","G"), ("G","C")}
    
    def is_valid_biallelic(row):
        r, a = row["ref"], row["alt"]
        if r not in valid_alleles or a not in valid_alleles:
            return False
        if (r,a) in ambiguous_pairs:
            return False
        return True
    
    df["is_valid"] = df.apply(is_valid_biallelic, axis=1)
    filtered_df = df[df["is_valid"]].copy()
    
    print(f"Original variants: {len(df)}, Kept valid: {len(filtered_df)}, Dropped: {len(df)-len(filtered_df)}")
    
    # -----------------------------
    # 3. Orient alleles (ALT = expression-increasing)
    # -----------------------------
    def orient_alt(row):
        if row["nes"] >= 0:
            return row["ref"], row["alt"]
        else:
            return row["alt"], row["ref"]
    
    filtered_df["ref_oriented"], filtered_df["alt_oriented"] = zip(*filtered_df.apply(orient_alt, axis=1))
    
    # -----------------------------
    # 4. Prepare PLINK input
    # -----------------------------
    plink_df = filtered_df[["snp_id","CHR","BP","pValue","ref_oriented","alt_oriented","tissue"]].copy()
    plink_df.columns = ["ID","CHR","BP","P","REF","ALT","tissue"]
    
    return plink_df

In [36]:
def process_snps_gtex(clump_output, tissues, p_cutoff=0.05, output_dir="data/gtex_output/21snps_alligned_across_tissues"):
    """
    Query GTEx for a list of SNPs, transform results, clean, and prepare PLINK input.
    
    Parameters
    ----------
    clump_output : df with SNPs to query. Must include 'snp_id' and 'gencodeId'
    tissues_list : list of tissue names to query
    p_cutoff : p-value threshold for significant cis-eQTLs.
    output_dir : str, optional

    
    Returns
    -------
    final_df : PLINK-ready, cleaned DataFrame
    """
    
    all_results = {}
    
    # 1. Query GTEx for each SNP
    for idx, row in clump_output.iterrows():
        # print(f"Running SNP {row['snp_id']}...")
        out = gtex_lookup_snp(
            gencodeId=row["gencodeId"],
            snp_id=row["snp_id"],
            tissues=tissues,
            p_cutoff=p_cutoff
        )
        all_results[row["snp_id"]] = out
    
    # 2. Transform results to long format
    rows = []
    for snp, info in all_results.items():
        for tissue, vals in info["significant_tissues"].items():
            vals2 = vals.copy()
            vals2["snp_id"] = snp
            vals2["gencodeId"] = info["gencodeId"]
            vals2["tissue"] = tissue
            rows.append(vals2)
    
    df_long = pd.DataFrame(rows)
    print(f"Number of unique SNPs: {df_long['snp_id'].nunique()}")
    print(f"Total rows: {len(df_long)}")
    
    p_cutoff_label = str(p_cutoff).replace("0.", "p0")
    
    # # 3. Save raw GTEx output
    # raw_file = f"{output_dir}/gtex_results_p_cutoff_label}_{'_'.join(tissues)}.csv"
    # df_long.to_csv(raw_file, index=False)
    # print(f"Raw GTEx results saved to {raw_file}")
    
    # 4. Clean data and allign according to NES
    plink_df = prepare_plink_input(df_long)
    
    # 5. Save cleaned PLINK-ready data
    cleaned_file = f"{output_dir}/gtex_results_{p_cutoff_label}_{'_'.join(tissues)}_cleaned.csv"
    plink_df.to_csv(cleaned_file, index=False)
    print(f"Cleaned PLINK-ready data saved to {cleaned_file}")
    
    return plink_df

### 1. Import list of SNPS

In [38]:

clump_output = pd.read_csv("data/clump_results_by_tissue/clump_index_snps_MERGED.tsv", sep="\t")
clump_output = clump_output.rename(columns={"ID": "snp_id"})

# Map gene - gencodeId
gene_to_gencode = {
    "CALCA": "ENSG00000110680",
    "CALCB": "ENSG00000175868",
    "CALCRL": "ENSG00000064989"
}

clump_output["gencodeId"] = clump_output["gene"].map(gene_to_gencode)
print("Number of unique snps = ",clump_output['snp_id'].nunique())
clump_output

Number of unique snps =  21


,snp_id,CHR,BP,P,ALT,tissue,gene,gencodeId
0,rs17567502,11,14551715,7.754948e-05,C,Nerve_Tibial,CALCA,ENSG00000110680
1,rs10766197,11,14900334,2.120945e-10,G,Brain_Cerebellum,CALCB,ENSG00000175868
2,rs10766197,11,14900334,2.752509e-13,G,Brain_Cerebellar_Hemisphere,CALCB,ENSG00000175868
3,rs2101888,11,14932569,1.450101e-05,T,Liver,CALCB,ENSG00000175868
4,rs12364639,11,15392110,7.596701e-07,A,Pancreas,CALCB,ENSG00000175868
5,rs6486214,11,14958830,3.080432e-05,G,Skin_Not_Sun_Exposed_Suprapubic,CALCB,ENSG00000175868
6,rs10741657,11,14893332,8.422995e-05,A,Testis,CALCB,ENSG00000175868
7,rs75567272,11,15104716,4.541004e-05,T,Testis,CALCB,ENSG00000175868
8,rs1583765,2,188106173,9.047480e-05,A,Thyroid,CALCRL,ENSG00000064989
9,rs112158314,2,187697790,2.789620e-05,G,Brain_Nucleus_accumbens_basal_ganglia,CALCRL,ENSG00000064989


#### 2. Select tissues of interest and apply process_snps_gtex to obtain the tissue-specific eQTL expression profile

In [27]:
tissue = clump_output["tissue"].unique()
tissue

array(['Nerve_Tibial', 'Brain_Cerebellum', 'Brain_Cerebellar_Hemisphere',
       'Liver', 'Pancreas', 'Skin_Not_Sun_Exposed_Suprapubic', 'Testis',
       'Thyroid', 'Brain_Nucleus_accumbens_basal_ganglia',
       'Artery_Coronary', 'Muscle_Skeletal', 'Whole_Blood',
       'Artery_Tibial', 'Colon_Sigmoid', 'Artery_Aorta', 'Ovary',
       'Adipose_Visceral_Omentum', 'Adipose_Subcutaneous'], dtype=object)

In [22]:
tissues = [['Brain_Cerebellum'], ['Brain_Cerebellar_Hemisphere'],
       ['Artery_Coronary'], ['Whole_Blood'],
       ['Artery_Tibial'], [ 'Artery_Aorta'],['Nerve_Tibial']]

# tissues = [['Testis']]
# tissues = [['Adipose_Visceral_Omentum'], ['Adipose_Subcutaneous'],['Artery_Coronary'],['Nerve_Tibial'], 
#        ['Liver'], ['Pancreas'], ['Skin_Not_Sun_Exposed_Suprapubic'], ['Testis'],
#        ['Thyroid'], ['Brain_Nucleus_accumbens_basal_ganglia'],
#       ['Muscle_Skeletal'], 
#        ['Colon_Sigmoid'],  ['Ovary'],
#       ['Brain_Cerebellum'], ['Brain_Cerebellar_Hemisphere'],
#        ['Artery_Coronary'], ['Whole_Blood'],
#        ['Artery_Tibial'], [ 'Artery_Aorta'], ['Brain_Hypothalamus'], ['Skin_Sun_Exposed_Lower_leg']]


for tissue in tissues:
    plink_df = process_snps_gtex(clump_output, tissue, p_cutoff=1)# p_cutoff=0.05

Number of unique SNPs: 21
Total rows: 21
Original variants: 21, Kept valid: 21, Dropped: 0
Cleaned PLINK-ready data saved to data/gtex_output/21snps_alligned_across_tissues/gtex_results_1_Brain_Cerebellum_cleaned.csv
Number of unique SNPs: 21
Total rows: 21
Original variants: 21, Kept valid: 21, Dropped: 0
Cleaned PLINK-ready data saved to data/gtex_output/21snps_alligned_across_tissues/gtex_results_1_Brain_Cerebellar_Hemisphere_cleaned.csv
Number of unique SNPs: 15
Total rows: 15
Original variants: 15, Kept valid: 15, Dropped: 0
Cleaned PLINK-ready data saved to data/gtex_output/21snps_alligned_across_tissues/gtex_results_1_Artery_Coronary_cleaned.csv
Number of unique SNPs: 14
Total rows: 14
Original variants: 14, Kept valid: 14, Dropped: 0
Cleaned PLINK-ready data saved to data/gtex_output/21snps_alligned_across_tissues/gtex_results_1_Whole_Blood_cleaned.csv
Number of unique SNPs: 15
Total rows: 15
Original variants: 15, Kept valid: 15, Dropped: 0
Cleaned PLINK-ready data saved to da

### 3. Format data for UP SET plot

In [298]:
folder = Path("data/gtex_output/21snps_alligned_across_tissues/")

dfs = []

for file in folder.glob("gtex_results_p1_*"):
    df = pd.read_csv(file)  
   
    # keep only required columns
    df = df[["ID", "ALT", "tissue"]]
    
    # get tissue name (first value)
    tissue_name = df["tissue"].iloc[0]
    
    # rename ALT -> tissue name
    df = df.rename(columns={"ALT": tissue_name})
    
    # drop tissue column (now encoded in column name)
    df = df.drop(columns="tissue")
    
    dfs.append(df)

# merge all dataframes on ID
merged = reduce(lambda x, y: x.merge(y, on="ID", how="outer"), dfs)
merged = merged[["ID"] + sorted(c for c in merged.columns if c != "ID")]
merged = merged.dropna(subset=["Brain_Cerebellum"])
merged.to_csv("data/gtex_output/21snps_alligned_across_tissues/merged_21snps_p1.csv", index=False)
merged

,ID,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Artery_Aorta,Artery_Coronary,Artery_Tibial,Brain_Cerebellar_Hemisphere,Brain_Cerebellum,Brain_Hypothalamus,Brain_Nucleus_accumbens_basal_ganglia,...,Liver,Muscle_Skeletal,Nerve_Tibial,Ovary,Pancreas,Skin_Not_Sun_Exposed_Suprapubic,Skin_Sun_Exposed_Lower_leg,Testis,Thyroid,Whole_Blood
0,rs10741657,NaN,NaN,G,NaN,NaN,A,A,G,G,...,A,NaN,NaN,NaN,A,A,NaN,A,A,NaN
1,rs10766197,NaN,NaN,G,NaN,NaN,G,G,A,A,...,G,NaN,NaN,NaN,G,G,NaN,G,A,NaN
2,rs10931283,C,T,C,C,C,T,T,T,T,...,C,C,C,C,C,T,C,C,T,T
3,rs112158314,A,G,G,G,G,G,A,G,G,...,A,G,G,G,A,A,A,A,G,G
4,rs12364639,NaN,NaN,A,NaN,NaN,G,A,G,A,...,G,NaN,NaN,NaN,A,A,NaN,G,A,NaN
5,rs147032266,G,G,G,G,G,A,A,A,A,...,G,A,G,G,G,A,A,G,G,A
6,rs1583765,C,A,C,C,C,A,A,C,C,...,C,A,C,C,A,A,A,A,A,A
7,rs17567502,C,C,C,T,C,C,C,C,C,...,C,C,C,T,T,NaN,NaN,T,C,NaN
8,rs1912850,G,A,G,G,G,A,A,A,A,...,G,G,G,G,A,A,G,G,G,A
9,rs2101888,NaN,NaN,C,NaN,NaN,T,T,T,C,...,T,NaN,NaN,NaN,T,T,NaN,T,C,NaN
